# VisionBridge — train real base_model.pt

One straight-line path: Install → Restart → Verify → Clone → Find dataset →
Prepare videos → Extract keypoints → Build metadata → Validate dataset →
Validate model shapes → Train → Validate outputs → Download.

Dataset: **ISL-CSLTR** (Kaggle `drblack00/isl-csltr-indian-sign-language-dataset`),
real sentence-level video clips (`Videos_Sentence_Level`, 687 mp4s).
`DATA_MODE = "video"` is fixed in STEP 5 — this notebook does not ask you to
choose a mode. A word-frames fallback (for a mirror that ships only
isolated-word jpgs instead of real videos) lives in an appendix at the very
end, out of the main path — OPTIONAL, only relevant if STEP 5 can't find a
video directory.

Feature contract enforced throughout (extraction, dataset, model all agree):
**pose = 132 dims (33 landmarks × 4), face = 1404 dims (468 landmarks × 3)**.

Each STEP below is one or two cells. Run top to bottom. If a cell prints
`STOP —`, fix the named problem before continuing — don't run further cells.
Extraction (STEP 7) is resumable and fault-tolerant: re-running it only
processes new/failed clips, and one bad video doesn't abort the rest.

**GPU**: no API exists to flip Colab's accelerator dropdown from inside a
notebook — do it once, manually, before running anything: **Runtime > Change
runtime type > T4 GPU > Save**. STEP 3 verifies it actually took.

## STEP 1 — Install extraction/training dependencies

RUN THIS CELL

In [ ]:
# Verified-working combo for MediaPipe Holistic extraction on Python 3.12 /
# Colab: mediapipe==0.10.21 + protobuf==4.25.9 + numpy==1.26.4. Not installing
# tensorflow — nothing in this pipeline needs it (extract_keypoints.py stubs
# out mediapipe's own unrelated tensorflow import instead of requiring it).
!pip uninstall -y mediapipe protobuf tensorflow tensorflow-cpu -q

!pip install -q --no-cache-dir \
    "numpy==1.26.4" \
    "protobuf==4.25.9" \
    "mediapipe==0.10.21" \
    "opencv-python-headless" \
    "pandas" \
    "kagglehub"

print("install done.")

## STEP 2 — Runtime restart

**STOP: Restart the Colab runtime now.** Runtime > Restart session.

After restarting, continue from STEP 3. Do not run STEP 3 before restarting
— the just-installed package versions aren't loaded into this process
until you do.

## STEP 3 — Verify environment

RUN THIS CELL (after restarting)

In [ ]:
import numpy
import google.protobuf
import mediapipe as mp

print("NumPy:", numpy.__version__)
print("protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("Has solutions:", hasattr(mp, "solutions"))

assert hasattr(mp, "solutions"), (
    "STOP — mediapipe has no .solutions. Either STEP 1 didn't finish, or the "
    "runtime wasn't restarted. Re-run STEP 1, restart, re-run STEP 3."
)

with mp.solutions.holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
) as holistic:
    print("Holistic initialized OK")

import torch
gpu_ok = torch.cuda.is_available()
print("GPU available:", gpu_ok)
print("GPU name:", torch.cuda.get_device_name(0) if gpu_ok else "none")
if not gpu_ok:
    print("STOP (recommended) — no GPU. Runtime > Change runtime type > T4 GPU > "
          "Save, then Runtime > Restart session, then re-run from STEP 3. Training "
          "will still run on CPU if you skip this, just much slower.")

## STEP 4 — Clone/load repository

RUN THIS CELL

In [ ]:
import os

BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_ROOT = os.path.join(BASE_DIR, "VisionBridge")

if not os.path.isdir(REPO_ROOT):
    os.chdir(BASE_DIR)
    !git clone https://github.com/BharathWaj-K-R/VisionBridge.git
elif not os.path.isfile(os.path.join(REPO_ROOT, "README.md")):
    # exists but looks corrupted/incomplete — nuke and reclone
    !rm -rf {REPO_ROOT}
    os.chdir(BASE_DIR)
    !git clone https://github.com/BharathWaj-K-R/VisionBridge.git
else:
    # existing clean clone — update it. --ff-only fails loudly on any
    # non-fast-forward state instead of silently running stale code.
    !git -C {REPO_ROOT} pull --ff-only

os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

script = open("backend/scripts/extract_keypoints.py", encoding="utf-8").read()
assert "POSE_FEATURE_DIM = 33 * 4" in script and "FACE_FEATURE_DIM = 468 * 3" in script, (
    "STOP — extract_keypoints.py is an older revision (missing the 132/1404 "
    "dimension fix). git pull failed silently or REPO_ROOT points somewhere "
    "unexpected. Check the output above."
)
print("extract_keypoints.py revision OK (132/1404 dims present)")

## STEP 5 — Locate dataset

RUN THIS CELL. Does not guess silently — if it can't find exactly one
sentence-video directory, it stops and prints the candidates it did find so
you can set `VIDEO_ROOT` yourself.

In [ ]:
import glob

DATA_MODE = "video"

_kaggle_mounts = glob.glob("/kaggle/input/isl-csltr-indian-sign-language-dataset*")
if _kaggle_mounts:
    dataset_path = _kaggle_mounts[0]
    print("using existing Kaggle input mount (no download):", dataset_path)
else:
    import kagglehub
    dataset_path = kagglehub.dataset_download("drblack00/isl-csltr-indian-sign-language-dataset")
    print("downloaded to:", dataset_path)

_candidates = [
    d for d in glob.glob(os.path.join(dataset_path, "**", "*Sentence_Level*"), recursive=True)
    if os.path.isdir(d) and "Video" in os.path.basename(d)
]

if len(_candidates) == 1:
    VIDEO_ROOT = _candidates[0]
    print("VIDEO_ROOT (auto-detected):", VIDEO_ROOT)
else:
    print(f"STOP — found {len(_candidates)} candidate video directories, expected exactly 1:")
    for c in _candidates:
        print(" ", c)
    print("Set VIDEO_ROOT manually to one of the above (or another real path), then re-run.")
    VIDEO_ROOT = None
    assert VIDEO_ROOT is not None, "Set VIDEO_ROOT above based on the candidates printed."

## STEP 6 — Prepare raw videos and labels

RUN THIS CELL

In [ ]:
import glob, os, shutil
import pandas as pd

os.chdir(REPO_ROOT)

RAW_VIDEOS_DIR = "data/raw_videos"
LABELS_CSV = "data/labels/ISLTranslate.csv"

shutil.rmtree(RAW_VIDEOS_DIR, ignore_errors=True)
os.makedirs(RAW_VIDEOS_DIR, exist_ok=True)
os.makedirs("data/labels", exist_ok=True)

video_files = []
for ext in ("mp4", "MP4", "avi", "AVI", "mov", "MOV"):
    video_files += glob.glob(os.path.join(VIDEO_ROOT, "**", f"*.{ext}"), recursive=True)

assert len(video_files) > 0, f"STOP — no video files found under VIDEO_ROOT: {VIDEO_ROOT}"
print(f"found {len(video_files)} video files")

# ADJUST if the sentence text isn't the immediate parent folder name
def sentence_id_from_path(video_path: str) -> str:
    return os.path.basename(os.path.dirname(video_path))

rows = []
for i, vp in enumerate(video_files):
    uid = f"clip{i:04d}"
    text = sentence_id_from_path(vp).replace("_", " ").strip()
    if not text:
        continue
    dst = os.path.join(RAW_VIDEOS_DIR, f"{uid}.mp4")
    shutil.copy(vp, dst)
    rows.append({"uid": uid, "text": text})

df = pd.DataFrame(rows)
df.to_csv(LABELS_CSV, index=False)

assert {"uid", "text"} <= set(df.columns), "STOP — CSV missing uid/text columns"
n_videos = len(glob.glob(os.path.join(RAW_VIDEOS_DIR, "*.mp4")))
assert len(df) == n_videos, (
    f"STOP — CSV has {len(df)} rows but {n_videos} videos were copied. Mismatch."
)
assert df["text"].str.len().min() > 0, "STOP — some rows have empty text."

print(f"videos == {n_videos}, CSV rows == {len(df)}")
print("\nsample rows:")
display(df.sample(min(5, len(df))))

## STEP 7 — Extract keypoints

RUN THIS CELL. Resumable and fault-tolerant — safe to re-run after an
interruption; already-valid `pose/<uid>.npy` + `face/<uid>.npy` pairs are
skipped, one bad clip is logged (not fatal), and both `ISLTranslate.csv`
(only successfully extracted, dimension-checked uids — no manual copy step
needed) and `extraction_failures.csv` are written and flushed to disk
immediately after each video, not just at the end — so a kill mid-run still
leaves an accurate completion record.

In [ ]:
os.chdir(REPO_ROOT)
!python backend/scripts/extract_keypoints.py \
  --videos_dir data/raw_videos \
  --labels_csv data/labels/ISLTranslate.csv \
  --out_dir data/processed/isltranslate

## STEP 8 — Build processed metadata

RUN THIS CELL. STEP 7 already wrote
`data/processed/isltranslate/ISLTranslate.csv` itself (the validated
manifest — only uids with dimension-checked pose+face files). This cell
just confirms that happened and shows how many rows survived vs the
original 687.

In [ ]:
os.chdir(REPO_ROOT)
import pandas as pd

processed_csv = "data/processed/isltranslate/ISLTranslate.csv"
assert os.path.isfile(processed_csv), (
    "STOP — processed CSV missing. Check STEP 7's output for errors."
)

original_df = pd.read_csv("data/labels/ISLTranslate.csv")
processed_df = pd.read_csv(processed_csv)
failures_path = "data/processed/isltranslate/extraction_failures.csv"
n_failures = len(pd.read_csv(failures_path)) if os.path.isfile(failures_path) else 0

print(f"original rows:  {len(original_df)}")
print(f"valid extracted: {len(processed_df)}")
print(f"failures logged: {n_failures}")

assert len(processed_df) > 0, "STOP — 0 valid extracted examples. Check extraction_failures.csv."

## STEP 9 — Dataset validation

RUN THIS CELL. Training does not start unless this passes — instantiates
the repo's actual `ISLTranslateKeypointDataset` and checks one real
example's shapes against the 132/1404 contract.

In [ ]:
import glob, sys
import pandas as pd

os.chdir(REPO_ROOT)

processed_csv = "data/processed/isltranslate/ISLTranslate.csv"
pose_files = glob.glob("data/processed/isltranslate/pose/*.npy")
face_files = glob.glob("data/processed/isltranslate/face/*.npy")

print("CSV rows:", len(pd.read_csv(processed_csv)))
print("Pose files:", len(pose_files))
print("Face files:", len(face_files))

assert len(pose_files) > 0, "STOP — zero pose files."
assert len(face_files) > 0, "STOP — zero face files."

pose_uids = {os.path.splitext(os.path.basename(p))[0] for p in pose_files}
face_uids = {os.path.splitext(os.path.basename(p))[0] for p in face_files}
mismatched = pose_uids ^ face_uids
if mismatched:
    print(f"STOP — {len(mismatched)} UIDs have pose but not face (or vice versa): "
          f"{sorted(mismatched)[:10]}...")
    raise AssertionError("pose/face UID mismatch")

sys.path.insert(0, "backend")
from app.training.isltranslate import ISLTranslateKeypointDataset
from app.models.base_model import POSE_INPUT_DIM, FACE_INPUT_DIM

dataset = ISLTranslateKeypointDataset("data/processed/isltranslate")
print(f"\nUsable examples: {len(dataset)}")
assert len(dataset) > 0, "STOP — 0 usable examples."

example = dataset[0]
print("\nUID:  ", example["uid"])
print("TEXT: ", example["text"])
print("POSE SHAPE:", tuple(example["pose"].shape))
print("FACE SHAPE:", tuple(example["face"].shape))

assert example["pose"].shape[-1] == POSE_INPUT_DIM, (
    f"STOP — pose feature dim {example['pose'].shape[-1]} != {POSE_INPUT_DIM}"
)
assert example["face"].shape[-1] == FACE_INPUT_DIM, (
    f"STOP — face feature dim {example['face'].shape[-1]} != {FACE_INPUT_DIM}"
)
print("\nshape contract OK")

## STEP 10 — Model shape validation

RUN THIS CELL. Runs exactly one real batch through the model — no optimizer
step — before committing to a full training run. Catches shape mismatches
in seconds instead of after minutes of wasted training time.

In [ ]:
import sys
import torch
from torch.utils.data import DataLoader

os.chdir(REPO_ROOT)
sys.path.insert(0, "backend")
from app.models.base_model import VisionBridgeBaseModel, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer, collate_ctc_batch

tokenizer = SimpleCharTokenizer()
dataset = ISLTranslateKeypointDataset("data/processed/isltranslate", tokenizer=tokenizer)
loader = DataLoader(dataset, batch_size=min(4, len(dataset)), shuffle=True, collate_fn=collate_ctc_batch)
batch = next(iter(loader))

print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Pose batch shape:", tuple(batch["pose"].shape))
print("Face batch shape:", tuple(batch["face"].shape))
print("Maximum batch temporal length:", batch["pose"].shape[1], "(<=", MAX_SEQUENCE_LENGTH, ")")
print("Input lengths:   ", batch["input_lengths"].tolist())
print("Label lengths:   ", batch["label_lengths"].tolist())

assert batch["pose"].shape[-1] == POSE_INPUT_DIM, "STOP — pose batch dim mismatch"
assert batch["face"].shape[-1] == FACE_INPUT_DIM, "STOP — face batch dim mismatch"
# Clips longer than MAX_SEQUENCE_LENGTH (e.g. the 4500-frame outlier clip0076)
# are uniformly downsampled by collate_ctc_batch before reaching the model —
# this must always hold, or the positional embedding will crash on long clips.
assert batch["pose"].shape[1] <= MAX_SEQUENCE_LENGTH, "STOP — pose temporal length exceeds MAX_SEQUENCE_LENGTH"
assert batch["face"].shape[1] <= MAX_SEQUENCE_LENGTH, "STOP — face temporal length exceeds MAX_SEQUENCE_LENGTH"
assert (batch["input_lengths"] <= MAX_SEQUENCE_LENGTH).all(), "STOP — an input_length exceeds MAX_SEQUENCE_LENGTH"

model = VisionBridgeBaseModel(vocab_size=tokenizer.vocab_size)
model.eval()
with torch.no_grad():
    logits = model(batch["pose"], batch["face"])
print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(batch["labels"].shape))

log_probs = torch.nn.functional.log_softmax(logits, dim=-1).transpose(0, 1)
loss_fn = torch.nn.CTCLoss(blank=0, zero_infinity=True)
loss = loss_fn(log_probs, batch["labels"], batch["input_lengths"], batch["label_lengths"])
print("CTC loss on this batch:", float(loss.item()))

assert torch.isfinite(loss), "STOP — CTC loss is not finite on the sanity batch."
print("\nmodel forward pass + CTC loss OK — safe to train")

## STEP 11 — Train

RUN THIS CELL

In [ ]:
%cd {REPO_ROOT}

!PYTHONPATH=backend python -m app.training.train_base_model \
  --data-dir data/processed/isltranslate \
  --output backend/app/models/weights/base_model.pt \
  --epochs 15 \
  --batch-size 4

## STEP 12 — Validate outputs

RUN THIS CELL

In [ ]:
import os

os.chdir(REPO_ROOT)
PT_PATH = "backend/app/models/weights/base_model.pt"
VOCAB_PATH = "backend/app/models/weights/base_model.vocab.json"

assert os.path.isfile(PT_PATH), f"STOP — {PT_PATH} does not exist. Training did not complete."
assert os.path.isfile(VOCAB_PATH), f"STOP — {VOCAB_PATH} does not exist."

print(f"{PT_PATH}: {os.path.getsize(PT_PATH) / 1e6:.2f} MB")
print(f"{VOCAB_PATH}: {os.path.getsize(VOCAB_PATH) / 1e3:.2f} KB")

import sys
sys.path.insert(0, "backend")
from app.models.base_model import load_frozen_base_model

try:
    model = load_frozen_base_model(PT_PATH)
    print("\nmodel loaded OK:", type(model).__name__)
except Exception as e:
    print(f"\nSTOP — file exists but failed to load: {e}")
    raise

## STEP 13 — Download artifacts

RUN THIS CELL

In [ ]:
PT_PATH = "backend/app/models/weights/base_model.pt"
VOCAB_PATH = "backend/app/models/weights/base_model.vocab.json"

try:
    from google.colab import files
    files.download(PT_PATH)
    files.download(VOCAB_PATH)
except ImportError:
    print("On Kaggle: download these two files from the notebook's Output pane —")
    print(PT_PATH)
    print(VOCAB_PATH)

print("drop both into backend/app/models/weights/ in your local repo, then commit+push.")

---
## Appendix — OPTIONAL, isolated word fallback

Not part of the main path. Only relevant if STEP 5 couldn't find a real
sentence-video directory and you've confirmed only isolated-word jpgs exist
(some Kaggle mirrors of this dataset ship `Frames_Word_Level/` instead of
`Videos_Sentence_Level/` — the mount used in STEP 5 has the real videos, so
this shouldn't be needed for the current run).

In [ ]:
# Set WORD_FRAMES_ROOT yourself if you actually need this path.
WORD_FRAMES_ROOT = None  # e.g. f"{dataset_path}/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Frames_Word_Level"

if WORD_FRAMES_ROOT is None:
    print("skipped — set WORD_FRAMES_ROOT above to use this fallback")
else:
    import sys
    import numpy as np
    import cv2
    import mediapipe as mp

    # Same fixed feature contract as extract_keypoints.py.
    POSE_FEATURE_DIM = 33 * 4    # 132
    FACE_FEATURE_DIM = 468 * 3   # 1404

    if "tensorflow" not in sys.modules:
        try:
            import tensorflow  # noqa: F401
        except Exception:
            import types
            _fake_tf = types.ModuleType("tensorflow")
            _fake_tools = types.ModuleType("tensorflow.tools")
            _fake_docs = types.ModuleType("tensorflow.tools.docs")
            _fake_docs.doc_controls = types.SimpleNamespace(
                do_not_generate_docs=lambda f: f,
                for_subclass_implementers=lambda f: f,
                do_not_doc_inheritable=lambda f: f,
            )
            _fake_tf.tools = _fake_tools
            _fake_tools.docs = _fake_docs
            sys.modules["tensorflow"] = _fake_tf
            sys.modules["tensorflow.tools"] = _fake_tools
            sys.modules["tensorflow.tools.docs"] = _fake_docs

    out_dir = "data/processed/isltranslate"
    pose_dir = os.path.join(out_dir, "pose")
    face_dir = os.path.join(out_dir, "face")
    os.makedirs(pose_dir, exist_ok=True)
    os.makedirs(face_dir, exist_ok=True)

    mp_holistic = mp.solutions.holistic
    word_folders = sorted(
        d for d in os.listdir(WORD_FRAMES_ROOT)
        if os.path.isdir(os.path.join(WORD_FRAMES_ROOT, d))
    )
    print(f"found {len(word_folders)} word folders")

    rows = []
    with mp_holistic.Holistic(static_image_mode=True, model_complexity=1) as holistic:
        for word in word_folders:
            word_dir = os.path.join(WORD_FRAMES_ROOT, word)
            imgs = sorted(
                f for f in os.listdir(word_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            )
            if not imgs:
                continue
            pose_frames, face_frames = [], []
            for img_name in imgs:
                img = cv2.imread(os.path.join(word_dir, img_name))
                if img is None:
                    continue
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)
                if results.pose_landmarks:
                    pose_frames.append(np.array(
                        [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
                        dtype=np.float32).flatten())
                else:
                    pose_frames.append(np.zeros(POSE_FEATURE_DIM, dtype=np.float32))
                if results.face_landmarks:
                    face_frames.append(np.array(
                        [[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark],
                        dtype=np.float32).flatten())
                else:
                    face_frames.append(np.zeros(FACE_FEATURE_DIM, dtype=np.float32))
            if not pose_frames:
                continue
            uid = word.strip().lower().replace(" ", "_").replace("/", "_").replace("'", "")
            np.save(os.path.join(pose_dir, f"{uid}.npy"), np.stack(pose_frames))
            np.save(os.path.join(face_dir, f"{uid}.npy"), np.stack(face_frames))
            rows.append({"uid": uid, "text": word.strip().lower().replace("_", " ")})
            print(f"[{len(rows)}] {word} -> {len(pose_frames)} frames")

    import pandas as pd
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "ISLTranslate.csv"), index=False)
    print("done — now go back to STEP 9 to validate.")

---
## Utility — reset, start clean

Optional. Wipes everything this notebook wrote (raw videos, labels,
processed features, trained weights) but leaves the read-only dataset mount
untouched, so STEP 5 won't need to re-download anything after this.

In [ ]:
import shutil

os.chdir(REPO_ROOT)
for path in [
    "data/raw_videos",
    "data/labels",
    "data/processed",
    "backend/app/models/weights/base_model.pt",
    "backend/app/models/weights/base_model.vocab.json",
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print("removed dir:", path)
    elif os.path.isfile(path):
        os.remove(path)
        print("removed file:", path)
print("clean. re-run from STEP 5 onward.")